In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import sklearn

df = pd.read_csv("../data/processed/BRinf_clean.csv", index_col=0, parse_dates=True)

df.head()

,ipca_mom,igpm_mom,igpdi_mom,fgv_igp_unknown,igp10_mom,focus_ipca_12m_median,electricity_consumption_total,electricity_consumption_ind,bofa_economic_condition_idx,cni_capacity_util_diff,...,focus_top5_med_t12,focus_top5_med_t13,focus_med_t1_sq,focus_mean_t1,focus_mean_t1_sq,focus_std_t1,focus_med_t2_sq,focus_mean_t2,focus_mean_t2_sq,focus_std_t2
2003-03-31,1.23,1.53,1.66,4.593,1.58,9.58,24960.0,115.0,-0.35,-0.3,...,0.55,0.50,0.8836,0.93,0.8649,0.13,0.4900,0.73,0.5329,0.17
2003-04-30,0.97,0.92,0.41,1.173,1.24,8.88,25284.0,93.0,-0.15,-0.5,...,0.45,0.78,1.0000,0.97,0.9409,0.14,0.2500,0.52,0.2704,0.18
2003-05-31,0.61,-0.26,-0.67,-1.860,0.02,8.22,24785.0,285.0,-0.47,0.0,...,0.60,0.43,0.2500,0.50,0.2500,0.10,0.3025,0.55,0.3025,0.15
2003-06-30,-0.15,-1.00,-0.70,-1.949,-0.59,6.91,24889.0,-99.0,-0.73,0.1,...,0.51,0.55,0.0100,0.17,0.0289,0.17,1.1025,1.07,1.1449,0.21
2003-07-31,0.20,-0.42,-0.20,-0.544,-0.73,6.29,24536.0,38.0,-0.14,-0.5,...,0.50,0.97,0.0900,0.39,0.1521,0.28,0.3364,0.56,0.3136,0.18


In [2]:
from ipca_forecast.features import train_preparation

X_train, y_train, X_test, y_test = train_preparation(df, 3)

In [3]:
top25 = X_train.corrwith(y_train).sort_values(key=abs, ascending=False).iloc[:25]

top25

treasury_rev_import_tax      0.310674
unemployment_rate_imf       -0.302177
cni_capacity_utilization     0.294701
savings_deposits_total       0.268672
focus_top5_med_t3            0.230324
focus_med_t3                 0.229424
focus_med_t2                 0.224206
focus_top5_med_t9           -0.222797
focus_mean_t2                0.218953
focus_top5_med_t4            0.215924
pis_pasep_tax                0.213049
focus_med_t2_sq              0.212119
focus_top5_med_t6            0.202569
focus_mean_t2_sq             0.199633
focus_top5_med_t2            0.187702
focus_med_t4                 0.180637
trade_balance_fob            0.171861
focus_top5_med_t7            0.167813
central_gov_net_revenue      0.159020
unemployment_rate_ibge      -0.158992
social_contrib_net_profit    0.158725
treasury_gross_revenue       0.152416
unemployment_male           -0.151001
focus_med_t6                 0.146971
treasury_revenue_total       0.143341
dtype: float64

In [4]:
import itertools

groups = list(itertools.combinations(top25.index, 4))
groups[:4]

[('treasury_rev_import_tax',
  'unemployment_rate_imf',
  'cni_capacity_utilization',
  'savings_deposits_total'),
 ('treasury_rev_import_tax',
  'unemployment_rate_imf',
  'cni_capacity_utilization',
  'focus_top5_med_t3'),
 ('treasury_rev_import_tax',
  'unemployment_rate_imf',
  'cni_capacity_utilization',
  'focus_med_t3'),
 ('treasury_rev_import_tax',
  'unemployment_rate_imf',
  'cni_capacity_utilization',
  'focus_med_t2')]

In [5]:
from sklearn.linear_model import LinearRegression

predictions = []

for c in groups:
    cols = list(c)

    model = LinearRegression()
    model.fit(X_train[cols], y_train)
    pred = model.predict(X_test[cols])

    predictions.append(pred)




In [6]:
import numpy as np

prediction_array = np.array(predictions)

csr_predictions = prediction_array.mean(axis=0)

In [7]:
mae_csr = sklearn.metrics.mean_absolute_error(y_test, csr_predictions)

rmse_csr = sklearn.metrics.root_mean_squared_error(y_test, csr_predictions)

print(f"mae_csr = {mae_csr}\nrmse_csr = {rmse_csr}")

mae_csr = 0.20729649596880256
rmse_csr = 0.2708298617816253


In [8]:
from ipca_forecast.features import add_lags

X_train_lags, y_train_lags, X_test_lags, y_test_lags = train_preparation(add_lags(df, 4), 3)

In [11]:
top25_lags = X_train_lags.corrwith(y_train_lags).sort_values(key=abs, ascending=False).iloc[:25]
top25_lags

focus_top5_med_t4                   0.480099
focus_med_t4                        0.476656
focus_med_t5_lag1                   0.421138
focus_top5_med_t5_lag1              0.419675
focus_med_t3                        0.410689
focus_med_t6_lag2                   0.402317
focus_top5_med_t3                   0.401030
focus_top5_med_t4_lag1              0.397779
focus_med_t4_lag1                   0.393957
focus_top5_med_t5_lag2              0.368190
focus_top5_med_t6_lag2              0.355682
focus_med_t5                        0.350628
unemployment_male_lag1             -0.347429
focus_top5_med_t5                   0.346290
focus_med_t6_lag1                   0.339354
cni_capacity_utilization_lag1       0.339141
focus_med_t7_lag3                   0.336545
focus_med_t5_lag2                   0.334490
cni_capacity_utilization_lag2       0.326611
focus_top5_med_t6_lag1              0.316121
electricity_consumption_ind_lag3    0.313914
focus_med_t8_lag4                   0.313027
focus_top5

In [15]:
groups_lags = list(itertools.combinations(top25_lags.index, 4))
groups_lags[:2]

[('focus_top5_med_t4',
  'focus_med_t4',
  'focus_med_t5_lag1',
  'focus_top5_med_t5_lag1'),
 ('focus_top5_med_t4', 'focus_med_t4', 'focus_med_t5_lag1', 'focus_med_t3')]

In [16]:
predictions_lags = []

for c in groups_lags:
    columns = list(c)

    model1 = LinearRegression()
    model1.fit(X_train_lags[columns], y_train_lags)
    pred1 = model1.predict(X_test_lags[columns])
    predictions_lags.append(pred1)


In [18]:
prediction_lags_array = np.array(predictions_lags)

csr_lags = prediction_lags_array.mean(axis=0)

mae_lags_csr = sklearn.metrics.mean_absolute_error(y_test_lags, csr_lags)

rmse_lags_csr = sklearn.metrics.root_mean_squared_error(y_test_lags, csr_lags)

print(f"mae_lags_csr = {mae_lags_csr}\nrmse_lags_csr = {rmse_lags_csr}")

mae_lags_csr = 0.18456253591550476
rmse_lags_csr = 0.23896571740116848


In [19]:
from ipca_forecast.evaluation import compare_with_focus

compare_with_focus(add_lags(df,4), range(1,13))

,mae_focus,mae_lasso,mae_csr,rmse_focus,rmse_lasso,rmse_csr,ratio_lasso,ratio_csr
h,,,,,,,,
1,0.152,0.165,0.137,0.185,0.207,0.169,1.119,0.912
2,0.189,0.194,0.174,0.242,0.250,0.217,1.034,0.898
3,0.190,0.198,0.185,0.250,0.264,0.239,1.057,0.957
4,0.192,0.237,0.188,0.256,0.311,0.245,1.217,0.959
5,0.197,0.256,0.183,0.261,0.333,0.237,1.277,0.909
6,0.195,0.224,0.185,0.260,0.298,0.239,1.148,0.920
7,0.196,0.215,0.192,0.261,0.281,0.249,1.076,0.953
8,0.198,0.224,0.193,0.263,0.297,0.248,1.131,0.943
9,0.199,0.263,0.197,0.266,0.340,0.251,1.280,0.945
